# Cascade quarantine

Generated from `fabric/04-silver.yaml` — do not edit by hand.

A row rejected at one table orphans its children at every table below it, and nothing notices: the children are valid in isolation, so no rule fires and they travel on until a join in a later layer discards them silently.

This runs **after** every silver table is built. It cannot be part of a child's own build — a parent is often cleansed after its child, because an order header is corrected from its lines.

In [ ]:
from datetime import datetime, timezone

from ttfabric.cleansing import cascade_quarantine

CASCADES = [{'child': 'stg_permission_shared',
  'parent': 'stg_card_holder',
  'join_on': 'card_holder_guid',
  'reason': 'parent cardholder failed cleansing'},
 {'child': 'stg_user_security_scope',
  'parent': 'stg_user_security',
  'join_on': 'username',
  'reason': 'parent persona failed cleansing'},
 {'child': 'stg_reader_inst_mapping',
  'parent': 'stg_institution',
  'join_on': 'institution_id',
  'reason': 'parent institution failed cleansing'}]
QUARANTINE_SUFFIX = '_quarantine'

load_id = f"load_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
print(f"cascade run {load_id}")


In [ ]:
total = 0
for rule in CASCADES:
    result = cascade_quarantine(
        spark,
        child=rule['child'],
        parent=rule['parent'],
        join_on=rule['join_on'],
        reason=rule['reason'],
        load_id=load_id,
        quarantine_suffix=QUARANTINE_SUFFIX,
    )
    total += result['quarantined']
    print(f"  {result['child']:<20} <- {result['parent']:<14} "
          f"{result['quarantined']:>7,} row(s) quarantined")

# Zero is the healthy state once upstream is clean. It is reported
# rather than asserted: the rows are a defect to fix, not a reason
# to stop the layer that correctly identified them.
print(f"\n{total:,} row(s) cascaded in total")
